# 1 — Provision the sandbox

Builds the thing the experiment measures: one corpus, materialised twice, where
the **only** difference between the two copies is governance.

**This costs money.** BigQuery storage, Dataplex profile scans, and — if you
have one — a Looker instance. `scripts/cleanup.py` deletes everything created
here.

`make bootstrap` runs all of this non-interactively. This notebook walks the
same steps with the reasoning in between, calling the same functions, so the two
cannot drift apart.

### Before you start

```bash
uv sync
gcloud auth application-default login   # ADC only; this project reads no SA key
cp .env.example .env                    # fill in GOOGLE_CLOUD_PROJECT
make apis                               # enable the Google Cloud APIs
make identities                         # one-time, needs project IAM admin
```

`make identities` is deliberately outside this notebook. It edits project-level
IAM, which is the one privileged step here, and it should be reviewed as a shell
script rather than buried in a cell.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent
sys.path[:0] = [str(ROOT / "src"), str(ROOT / "scripts")]

from google.cloud import bigquery

import catalog_setup
import config
import corpus
import looker_check

import setup  # scripts/setup.py — the same steps `make setup` runs

print(f"project   {config.require_project()}")
print(f"BigQuery  {config.BQ_LOCATION}   Dataplex {config.DATAPLEX_LOCATION}")
print(f"tiers     {', '.join(config.TIER_LABELS[t] for t in config.TIERS)}")
print(f"tier SAs  {'ON' if config.USE_TIER_SA else 'OFF - the fence is not in force'}")

## The corpus is built to be hostile

A warehouse an agent finds easy proves nothing. Every table here carries at
least one trap that a competent analyst reading only the schema would fall into:

| Trap | Where | What it does to a naive query |
|---|---|---|
| T1 | `revenue_amount` | a decoy column: gross, not net, and simply wrong |
| T2 | `status_flg` | boolean whose `TRUE` means *refunded* |
| T3 | `is_active` | disagrees with the governed definition of "active" |
| T4 | `txn_amt_x2` | nullable, so `AVG` and `SUM` diverge |

The governed rules that resolve them are in `corpus.py` and are the text that
tier 1 publishes to the Knowledge Catalog.

In [ ]:
for table in corpus.CORPUS:
    print(f"{table.name:<20} {len(table.columns)} columns")
print()
for name, rule in corpus.GUIDELINES.items():
    print(f"{name}:\n  {rule}\n")

## Tables

Generate once, copy to every tier, then apply governance to the schemas —
descriptions on tier 1, descriptions explicitly *stripped* on tier 0. Stripping
matters: the control has to be provably ungoverned, not merely un-enriched by a
previous run.

Idempotent. Re-running is safe and is the normal way to repair a partial setup.

In [ ]:
client = bigquery.Client(project=config.require_project())
setup.provision_bigquery(client)

## Verify the control before trusting it

Two things have to hold, and neither is safe to assume:

1. **The traps are actually live** — the realized rates are printed against the
   targets in `corpus.py`. A generator that quietly produced no refunds would
   make T2 unmeasurable while everything still looked fine.
2. **The tiers agree on every golden value.** If they disagree, governance is
   not the only difference between them and every tier-1-minus-tier-0 number in
   the results is meaningless.

In [ ]:
assert setup.verify_corpus(client), "tiers disagree - the control is contaminated"

## Governance

Profile scans, then business rules as catalog aspects, then a business glossary
whose terms link to specific columns. Tier 0 gets its aspects stripped first,
because aspects are additive and a stale one from an earlier run would linger
beside the intended governance.

Catalog enrichment uses preview APIs and treats failures as **non-fatal**, so
read the output rather than the exit code. That is also why the rules are read
back below instead of assumed: writing an aspect can succeed while storing
nothing at all, because a field the template does not recognize is dropped
rather than rejected.

In [ ]:
catalog_setup.create_and_run_profile_scans()
for tier in config.TIERS:
    catalog_setup.strip_governance_aspects(tier)
catalog_setup.attach_business_rules()
catalog_setup.create_glossary_and_links()

assert setup.verify_governance(), "governance is not stored as designed"

## Looker (optional — Path 2 and `p4_looker_ca`)

Nothing here creates a Looker instance, and nothing in this repo will: that is
an annual commitment, and no setup script should be able to start one. If you
have an instance, `docs/looker_setup.md` covers the manual steps and
`make looker-plan` shows what would be created before anything is.

Without Looker, seven of the ten arms still run at both tiers — add
`SKIP_LOOKER=1` to any `make` target. The central governed-versus-ungoverned
result survives; you lose the semantic-layer path.

In [ ]:
looker_check.report()

## Prove the fence

**Do not skip this.** The tiers are separated by IAM, not by prompt, and the
reason is a leak that was measured rather than imagined: Knowledge Catalog
search is project-wide and content-addressed — `search_entries` takes a query,
not a scope — so a tier-0 agent asking for "revenue" was handed the tier-1
governed entry and answered from it. No MCP parameter prevents that. Catalog
search being ACL-filtered per caller does.

Every negative check below is paired with a positive control on the same tool,
because a call that fails for the wrong reason looks exactly like a fence
holding. An unproven negative reports `????`, not `PASS`.

IAM takes a minute or two to propagate and the catalog search index lags
further, so a failure straight after `make identities` is worth re-running
before believing.

In [ ]:
!cd .. && uv run python examples/verify_isolation.py

## Next

- **[`02_walkthrough.ipynb`](02_walkthrough.ipynb)** — watch one arm answer one
  question at both tiers, tool call by tool call.
- `make plan` — what a full sweep would cost in time and tokens. Free.
- `make smoke` — ten cells, one per arm, about ten minutes.

When you are done: `make teardown` deletes every resource created above.